# TODO-RESULT title

*2026-11-11. Week 02. Plan: `docs/plans/oracle_value.md`. **ORACLE ceiling**: reads true labels to compute the value stored in memory (and to file it). Builds on `week_01/2026-11-09_oracle_true_filing/oracle_true_filing.ipynb`, which closed "cleaner filing" and left the stored value open; does not repeat it.*

**Question.** Ramen's memory stores, for every image $j$, the entropy gradient $g_j$, which pushes toward $j$'s *zero-shot* class. If it stored instead the cross-entropy gradient toward $j$'s **true** class, filed under that class (so each queue still pushes toward its own class), would Ramen predict better? If even this oracle does not help, no label-free way of improving the stored values can.

**Hypothesis (pre-registered).** On CIFAR-10-C seed 0, the ORACLE is above the matched control *Ramen without self* by more than the CIFAR-10-C noise floor (0.18).

**Variants**: one run each, same stream / batch / seed / GPU model, so every image is paired across runs (`ramen_lab.methods.stored_value`)
| name | class | config (on top of `cfg/<ds>/Ramen.yaml`) | what it changes | uses labels? |
|---|---|---|---|---|
| Ramen | upstream `Ramen` | — | baseline (week-01 run `pc2_<ds>_Ramen_b<B>_s0`) | no |
| **Ramen without self** (control) | `RamenStoredValue` | `exclude_self: true` | a query does not retrieve its own memory entry | no |
| CE to zero-shot label (control) | `RamenStoredValue` | `stored_value: ce_pseudo`, `exclude_self: true` | value = CE gradient toward $\hat y_j$: the oracle's loss form, no label information | no |
| **ORACLE** | `RamenStoredValueOracle` | `stored_value: ce_true`, `filing: true_label`, `exclude_self: true` | value = CE gradient toward $y_j$, filed under $y_j$ | **ORACLE** |
| ORACLE, zero-shot filing (CIFAR only) | `RamenStoredValueOracle` | `stored_value: ce_true`, `exclude_self: true` | value = CE gradient toward $y_j$, filed under $\hat y_j$ (value without the filing) | **ORACLE** |

**Why "without self".** Ramen writes a batch into memory *before* retrieving, so every query retrieves its own entry at distance 0. With the oracle value that entry is the query's own true-label gradient, which would be label leakage, not a memory effect. So every compared variant drops the query's own entry, and the comparison is against Ramen with the same change.

**Data.** Severity 5, mixed stream (all domains shuffled together). CIFAR-10-C / CIFAR-100-C: 15 corruptions × 10,000 = 150,000 images, 10 / 100 classes. ImageNet-C (5k subset): 15 × 5,000 = 75,000 images, 1,000 classes. DomainNet: 6 domains, 586,575 images, 345 classes. **Backbones / batch (frozen, D18).** CIFAR-10-C ViT-B/32, 100; CIFAR-100-C ViT-B/16, 32; ImageNet-C ViT-B/16, 24; DomainNet ViT-B/32, 32. **Seed.** 0 (a seed changes only the stream order). **GPU.** RTX 3080 Ti (GPU 0), the model of the baseline runs.
**How produced.** `python oracle_value.py run CIFAR10C CIFAR100C --seed 0 --ram-gb 13`, then `run ImageNetC5K DomainNet --seed 0 --ram-gb 10` (with `CUDA_MODULE_LOADING=LAZY`), detached through `ramen_lab.launch` on GPU 0. Resources per run: section 2.
**What counts (pre-registered).** Δ = ORACLE − Ramen without self, against each dataset's floor (2 × range of Ramen seeds 0-4). Δ ≤ floor on CIFAR-10-C: better stored values have no ceiling; close "memory contents". Δ > floor: the value is a lever; the CE-to-zero-shot-label control then tells how much of it is label-free.

## How Ramen adapts, and what this experiment changes

For a batch $\{x_i\}_{i=1}^{B}$, zero-shot CLIP with LayerNorm affine parameters $\theta_0$:

**1. Embedding, prediction, entropy** (`src/methods/Ramen.py:114-116`, `src/methods/losses.py:4`)
$$z_i = f_{\theta_0}(x_i), \qquad p_i = \mathrm{softmax}(W^\top z_i), \qquad \hat y_i = \arg\max_c p_{ic}, \qquad H_i = -\sum_c p_{ic}\log p_{ic}$$

**2. Stored value**: a per-sample gradient w.r.t. that sample's own copy of the LayerNorm parameters (`Ramen.py:118-120`; ours `stored_value.py: RamenStoredValue.compute_values`)
$$v_i = \nabla_\theta\, L_i \big|_{\theta_0}, \qquad \text{Ramen: } L_i = H_i$$

**3. Memory write**, before retrieval (`Ramen.py:123-133`; ours `RamenStoredValue.write`): $(z_i, v_i, H_i, t_i)$ goes into queue $\mathcal M_{q_i}$, Ramen: $q_i = \hat y_i$. Capacity $K$ per class; a full queue replaces its lowest-priority entry ($t$ = arrival counter, fp16).

**4. Retrieval and aggregation** (`Ramen.py:135-154`; ours `RamenStoredValue.retrieve`): from each non-empty queue $c$ the $k$ entries nearest to $z_i$,
$$w_{ij} = e^{-H_j}\, e^{-\beta \lVert z_i - z_j \rVert_2}, \qquad G_i = \frac{1}{|\mathcal C^+|} \sum_{c \in \mathcal C^+} \sum_{j \in \mathrm{top}k_c(z_i)} w_{ij}\, v_j$$

**5. One SignSGD step per sample, predict, reset** (`Ramen.py:157-168`, `src/models/optimizer.py:100`; ours `RamenStoredValue.adapt_and_predict`)
$$\theta_i = \theta_0 - \eta\,\mathrm{sign}(G_i), \qquad \text{prediction } \arg\max_c\, \mathrm{softmax}(W^\top f_{\theta_i}(x_i))_c, \qquad \theta \leftarrow \theta_0$$

### The change
Steps 2-4, one term each (config keys `stored_value`, `filing`, `exclude_self`; registries `VALUES`, `FILINGS`):
$$\boxed{L_j = \mathrm{CE}(p_j, y_j)} \;(\texttt{ce\_true}), \qquad \boxed{q_j = y_j} \;(\texttt{true\_label}), \qquad \boxed{\mathrm{top}k_c(z_i) \text{ taken over } \mathcal M_c \setminus \{i\}} \;(\texttt{exclude\_self})$$
The control `ce_pseudo` uses $L_j = \mathrm{CE}(p_j, \hat y_j)$. The weights $w_{ij}$ keep the zero-shot entropy $H_j$ in every variant. $1/|\mathcal C^+|$ and any positive scale of the values leave $\mathrm{sign}(G_i)$ unchanged. Self-exclusion (`drop_self`) retrieves $k+1$ entries per queue and drops the query's own entry, or the $(k+1)$-th if the query is not among them.

### What it should do, and how it could fail
- With softmax outputs, $\nabla_{\text{logits}}\mathrm{CE}(p_j, y) = p_j - e_y$. The stored value pushes toward $y$ with a magnitude that grows as $1 - p_{jy}$: images that zero-shot gets **wrong** carry large values toward their true class, and confidently-right ones carry almost nothing. By contrast the entropy gradient of a wrong image pushes toward its wrong $\hat y_j$ (`grad_alignment.ipynb`, Finding 3).
- Filing by $y_j$ keeps Ramen's *push balance*: every entry of queue $c$ pushes toward $c$, so the active set has $k$ entries pushing toward every class. That is the property `oracle_true_filing` found to matter (paper §3.2 "prediction balance", App. B).
- Predicted [hypothesis]: on zero-shot-wrong queries, $G_i$ moves from the null (`grad_alignment.ipynb`, Finding 2) toward the true class, so they gain. The log measures this as the share of push mass on the true class (section 5).

How it could fail or mislead:
1. **A gradient is a direction for one image at $\theta_0$.** Applied to a different image $i$, it may not help, since neighbours share the domain, not the class (`knn_heldout`). Then the ceiling is small.
2. **Near-duplicates.** On CIFAR-10-C, CIFAR-100-C and ImageNet-C, one base image appears under 15 corruptions, so the memory can hold the query's own content, with its label, under another corruption. That inflates the ceiling there. DomainNet has no such duplicates.
3. **fp16.** Values are stored in fp16 without a loss scale (Ramen's regime). A non-finite value stops the run. Underflow (zero coordinates) is logged (section 2).

**Where labels enter.** Only in the two ORACLE runs, in $v_j$ and $q_j$ of *memory entries*. With `exclude_self`, the query's own entry is never in its active set (tested: `tests/test_lab_stored_value.py::test_retrieval_matches_a_manual_recomputation`), so $y_i$ does not reach $x_i$'s own step. A batch-mate's label can, as any memory entry's can.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, warnings
from pathlib import Path
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parents[2])); sys.path.insert(0, str(Path.cwd()))
from IPython.display import display
from ramen_lab.analysis.plots import use_style
import oracle_value as ov

use_style()
pd.set_option("display.width", 220, "display.max_columns", 30, "display.precision", 3)
DATASETS = [d for d in ("CIFAR10C", "CIFAR100C", "ImageNetC5K", "DomainNet") if ov.available(d, ov.MAIN) == list(ov.MAIN)]
RUNS = {d: ov.load_dataset(d) for d in DATASETS}
FLOORS = {d: ov.floor_of(d) for d in DATASETS}
print("datasets with all main runs:", [ov.DISPLAY[d] for d in DATASETS])
print("noise floors (2 x range of Ramen seeds 0-4):", {ov.DISPLAY[d]: round(f, 3) for d, f in FLOORS.items()})

## 1. The data

Images per domain, and NoAdapt (zero-shot) and Ramen errors (week-01 baseline runs, seed 0). The loader above already checked that every run saw the same images in the same order (`load_dataset` asserts equal `sample_idx`, `domain_idx`, `label`).

In [ ]:
for d in DATASETS:
    print(ov.DISPLAY[d])
    display(pd.DataFrame(ov.data_table(RUNS[d])).set_index("domain"))

## 2. Sanity checks

- **Identity.** The default `RamenStoredValue` equals upstream Ramen bit for bit, both in the test (`test_default_is_bit_identical_to_upstream_ramen`: a stream that fills and evicts) and on real data (`oracle_value.py check CIFAR10C --n 200`: identical predictions, 3,000 of 3,000 (100.00%)).
- **The changes did what they claim** (`tests/test_lab_stored_value.py`):
  - the retrieval with and without self-exclusion matches a direct recomputation;
  - `ce_true` is each image's own CE gradient;
  - with true-label filing, every entry sits in the queue of its true class and pushes toward it;
  - label-free settings ignore labels; the ORACLE's predictions change with them.
- **Below:** how much weight the query's own entry carries in Ramen's top-$k$ (what `exclude_self` removes), fp16 underflow of the stored values, and resources.

In [ ]:
display(pd.DataFrame([row for d in DATASETS for row in ov.sanity_table(RUNS[d])]).set_index(["dataset", "variant"]))

## 3. TODO-RESULT headline finding

In [ ]:
GAINS = {d: ov.gain_table(RUNS[d], FLOORS[d]) for d in DATASETS}
cols = ["acc", "gain", "ci_low", "ci_high", "floor", "beyond floor", "fixed", "broken"]
display(pd.DataFrame([r for d in DATASETS for r in GAINS[d]]).set_index(["dataset", "variant"])[cols])
fig = ov.gain_figure([r for d in DATASETS for r in GAINS[d]], "Gain over Ramen without self, seed 0")

## 4. TODO-RESULT zero-shot right / wrong finding

In [ ]:
SPLIT = [ov.zs_split_table(RUNS[d]) for d in DATASETS]
display(pd.DataFrame([r for t in SPLIT for r in t]).set_index(["dataset", "variant"])[["zero-shot right: correct", "zero-shot wrong: correct"]])
fig = ov.zs_split_figure(SPLIT)

## 5. TODO-RESULT mechanism finding

Push mass of entry $j$ in query $i$'s active set: $w_{ij}\lVert v_j\rVert$. Its push class is $\hat y_j$ for the entropy and CE-to-zero-shot-label values, and $y_j$ for the ORACLE values. The log keeps the 5 classes with the most push mass per query. A true class outside them counts as 0, a lower bound; the table gives how often that happens.

In [ ]:
display(pd.DataFrame([r for d in DATASETS for r in ov.mechanism_table(RUNS[d])]).set_index(["dataset", "variant", "images"]))
for d in DATASETS:
    fig = ov.mechanism_figure(RUNS[d])

## 6. TODO-RESULT stream finding

In [ ]:
fig = ov.stream_figure([RUNS[d] for d in DATASETS])

## Interpretation

TODO-RESULT

## Glossary

- **$z_i$**: L2-normalised CLIP image feature of image $i$ (the memory key). **$p_i$**: zero-shot class probabilities. **$\hat y_i$**: zero-shot prediction. **$y_i$**: true label (read only by the ORACLE runs).
- **$H_i$**: entropy of $p_i$; Ramen's loss and, through $e^{-H_j}$, the memory weight.
- **$v_j$ (stored value)**: the per-sample gradient written to memory. Ramen: entropy gradient. `ce_pseudo`: CE gradient toward $\hat y_j$. `ce_true` (ORACLE): CE gradient toward $y_j$.
- **Push class**: the class an entry's value makes the model more confident in ($\hat y_j$, or $y_j$ for `ce_true`). **Push mass**: $w_{ij}\lVert v_j\rVert$.
- **Filing / queue $q_j$**: the class queue an entry is written into. Ramen: $\hat y_j$.
- **Self / exclude_self**: the query's own memory entry (Ramen writes before retrieving, so it is always retrievable at distance 0); `exclude_self` skips it and takes the next-nearest entry.
- **$w_{ij}$, $G_i$**: retrieval weight and aggregated gradient (step 4). **$K, k, \beta, \eta$**: queue capacity, entries per queue, distance temperature, SignSGD step (`cfg/<ds>/Ramen.yaml`).
- **Noise floor**: 2 × (max − min) of Ramen's accuracy over seeds 0-4 at the fixed batch (`2026-11-01_baseline`). A gain counts only above it.
- **Paired gain, 95% CI**: accuracy difference on the same images in the same order; CI from a cluster bootstrap that resamples base images (all corruptions of one image together; single images on DomainNet).
- **Fixed / broken**: images the variant gets right and the reference gets wrong / the reverse.
- **ORACLE**: a run that reads true labels. A diagnostic ceiling, never a method. **Control**: a run with the same side change but without the idea.
- **Mixed stream**: all domains shuffled into one stream. **fp16 priority**: Ramen's arrival counter, stored in half precision (coarse beyond 2,048, frozen beyond 65,504).